1. Include the data set and preprocessing for following machine learning.

In [16]:
## import libs
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

##include minist dataset
mnist = fetch_openml('mnist_784', version=1)
print(mnist.keys())
X, y = mnist["data"], mnist["target"]
print(X.shape)
print(y.shape)

##Preprocessing
#alignment
X = X.to_numpy()
X = X.astype(np.float32)
y = y.astype(np.int64)
y = y.to_numpy()
#normalization
X = X / 255.0
# print(X[0])
# print(y[0])

##Split the data set into training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/7, random_state=0)

dict_keys(['data', 'target', 'frame', 'categories', 'feature_names', 'target_names', 'DESCR', 'details', 'url'])
(70000, 784)
(70000,)


2. Classifier Models Set up

In [25]:
## Minimum Euclidean Distance Classifier
class MED:
    def __init__(self):
        pass

    def fit(self, X, y):
        self.classes = np.unique(y)
        for w in self.classes:
            X_w = X[y == w]
            mean_w = np.mean(X_w, axis=0)
            if not hasattr(self, 'means'):
                self.means = mean_w[np.newaxis, :]
            else:
                self.means = np.vstack((self.means, mean_w))

    def predict(self, X):
        distances = np.linalg.norm(X[:, np.newaxis] - self.means, axis=2)
        return self.classes[np.argmin(distances, axis=1)]
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    
## Mnimun Mahalanobis Distance Classifier
class MMD:
    def __init__(self):
        pass

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.covariance = np.cov(X, rowvar=False) + 1e-6 * np.eye(X.shape[1])  
        self.inv_covariance = np.linalg.inv(self.covariance)
        for w in self.classes:
            X_w = X[y == w]
            mean_w = np.mean(X_w, axis=0)
            if not hasattr(self, 'means'):
                self.means = mean_w[np.newaxis, :]
            else:
                self.means = np.vstack((self.means, mean_w))

    def predict(self, X):
        diff = X[:, np.newaxis] - self.means
        distances = np.einsum('ijk,kl,ijl->ij', diff, self.inv_covariance, diff)
        return self.classes[np.argmin(distances, axis=1)]
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    
## Bayes Classifier
class Bayes:
    def __init__(self):
        pass

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors = []
        self.means = []
        self.covariances = []
        for w in self.classes:
            X_w = X[y == w]
            prior_w = X_w.shape[0] / X.shape[0]
            mean_w = np.mean(X_w, axis=0)
            cov_w = np.cov(X_w, rowvar=False) + 1e-6 * np.eye(X.shape[1])  
            self.priors.append(prior_w)
            self.means.append(mean_w)
            self.covariances.append(cov_w)
        self.priors = np.array(self.priors)
        self.means = np.array(self.means)
        self.covariances = np.array(self.covariances)

    def predict(self, X):
        log_likelihoods = []
        for w in range(len(self.classes)):
            mean_w = self.means[w] 
            cov_w = self.covariances[w]
            inv_cov_w = np.linalg.inv(cov_w)
            det_cov_w = np.linalg.det(cov_w)
            diff = X - mean_w
            log_likelihood_w = -0.5 * np.sum(diff @ inv_cov_w * diff, axis=1) - 0.5 * np.log(det_cov_w) + np.log(self.priors[w])
            log_likelihoods.append(log_likelihood_w)
        log_likelihoods = np.array(log_likelihoods).T
        return self.classes[np.argmax(log_likelihoods, axis=1)]
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)




3. Evaluate Classification

In [26]:
classifier_med=MED()
classifier_med.fit(X_train, y_train)
y_pred_med=classifier_med.predict(X_test)   
accuracy_med=classifier_med.score(X_test,y_test)
print("MED accuracy:", accuracy_med)

# classifier_bayes=Bayes()
# classifier_bayes.fit(X_train, y_train)
# y_pred_bayes=classifier_bayes.predict(X_test)
# accuracy_bayes=classifier_bayes.score(X_test,y_test)
# print("Bayes accuracy:", accuracy_bayes)

classifier_mmd=MMD()
classifier_mmd.fit(X_train, y_train)
y_pred_mmd=classifier_mmd.predict(X_test)
accuracy_mmd=classifier_mmd.score(X_test,y_test)
print("MMD accuracy:", accuracy_mmd)

MED accuracy: 0.805
MMD accuracy: 0.8488
